# Decode POCSAG

This notebook uses a local pager capture if you have one, otherwise it synthesizes a simple 2-FSK burst and walks through symbol recovery. The decoding is intentionally lightweight and focused on signal intuition.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


In [ ]:
CAPTURE_ROOT = ROOT / "assets" / "local"
capture_status = probe_rtlsdr()
display(Markdown(
    f"**RTL-SDR status:** installed={capture_status['installed']}, "
    f"available={capture_status['available']}. {capture_status['message']}"
))


In [ ]:
capture_path = CAPTURE_ROOT / "pocsag_iq.npz"
symbol_rate = 1200

if capture_path.exists():
    fs_iq, iq = load_complex_capture(capture_path)
    print(f"Loaded local capture: {capture_path.name}, fs={fs_iq}")
else:
    fs_iq = 96_000
    bits = np.array(([1, 0, 1, 0, 0, 1, 1, 0] * 40), dtype=int)
    samples_per_symbol = fs_iq // symbol_rate
    freq_steps = np.repeat(np.where(bits == 1, 2400, -2400), samples_per_symbol)
    phase = 2 * np.pi * np.cumsum(freq_steps) / fs_iq
    iq = np.exp(1j * phase)
    print("Using synthetic 2-FSK pager fallback.")


In [ ]:
discriminator = np.angle(iq[1:] * np.conj(iq[:-1])) * fs_iq / (2 * np.pi)
discriminator = np.concatenate([discriminator, discriminator[-1:]])
samples_per_symbol = fs_iq // symbol_rate
usable = len(discriminator) // samples_per_symbol
symbol_values = discriminator[: usable * samples_per_symbol].reshape(usable, samples_per_symbol).mean(axis=1)
decoded = (symbol_values > 0).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
plot_waveform(discriminator[:12000], fs=fs_iq, ax=axes[0], title="FSK discriminator output")
plot_spectrum(discriminator, fs=fs_iq, ax=axes[1], title="Baseband spectrum")
axes[1].set_xlim(0, 5000)
axes[1].set_ylim(-100, 5)
axes[2].plot(decoded[:100], drawstyle="steps-post")
axes[2].set_ylim(-0.2, 1.2)
axes[2].set_title("Decoded bits")
plt.tight_layout()

display(Markdown(f"**First 80 bits:** `{''.join(map(str, decoded[:80]))}`"))


## Key Takeaway

POCSAG-style FSK decoding starts the same way as many other digital radio tasks: convert frequency changes into a real-valued discriminator output, average over symbol periods, and then make decisions.